#### Libraries

In [20]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import random
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, GroupKFold

# Plotting style
sns.set_style('darkgrid')
sns.set_theme(font_scale=1.)

# Set the state (we will call the function when we need to make sure that we get the same results every time we run the code)
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_all_seeds(1234)

In [21]:
def get_basic_model(input_dim, hidden_dim, output_dim):
    # Assignment: Define the model usign the torch.nn.Sequential approach (see the exercise part for help)
    # YOUR CODE HERE
    return torch.nn.Sequential(
        torch.nn.Linear(in_features=input_dim, out_features=hidden_dim, bias=True),     # Input layer
        torch.nn.ReLU(),                                                                # Activation function
        torch.nn.Linear(in_features=hidden_dim, out_features=output_dim, bias=True),    # Output layer
    ) 


In [22]:
def fit_nn(model, X_train, y_train, X_test, y_test, criterion,n_epochs,lr):
    # Assignment: Define the 'optimizer' as stochastic gradient descent (SGD) method with the learning rate defined in lr.     
    optimizer = torch.optim.SGD(params=model.parameters(), lr=lr)
    # YOUR CODE HERE
    

    # Define a dictionary to store the loss values for each epoch
    results = {'train': [], 'test': []}

    # Training loop
    for epoch in range(n_epochs):    
        # Set the model to training mode
        model.train()
        # Make sure that the gradients are zero before you use backpropagation
        optimizer.zero_grad()
        # Make a forward pass through the model to compute the outputs
        outputs = model(X_train)
        # Compute the loss
        loss = criterion(outputs, y_train)
        # Do a backward pass to compute the gradients wrt. model parameters using backpropagation.
        loss.backward()
        # Update the model parameters by making the optimizer take a gradient descent step
        optimizer.step()

        with torch.no_grad(): # No need to compute gradients for the validation set
            # Set the model to evaluation mode
            model.eval()  
            # Compute the loss for the test set
            test_outputs = model(X_test)
            test_loss = criterion(test_outputs, y_test)
        
        # Store the training and test loss for this epoch in the dictionary
        results['train'].append(loss.item())
        results['test'].append(test_loss.item())

        # Print the loss every 1000 epochs
        if (epoch+1) % 1000 == 0:
            print(f'Epoch [{epoch+1}/{n_epochs}], Training loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}')

    return model, results

In [23]:
df = pd.read_csv('HR_data.csv')

hr_cols = ['HR_Mean', 'HR_Median', 'HR_Min', 'HR_Max', 'HR_AUC']
group_cols = ['Round', 'Individual']   

baseline = df[df['Phase'] == 'phase1'].set_index(group_cols)[hr_cols]
baseline = baseline.rename(columns={c: c + '_baseline' for c in hr_cols})

df = df[df['Phase'].isin(['phase2', 'phase3'])].copy()
df = df.merge(baseline, on=group_cols, how='left')

for c in hr_cols:
    df[c] = df[c] - df[c + '_baseline']

df = df.drop(columns=[c + '_baseline' for c in hr_cols])


X = df.drop(columns=['Cohort', 'Round', 'Frustrated'])
X = X.iloc[:, 1:]
X['Phase'] = X['Phase'].map({'phase1': 1, 'phase2': 2, 'phase3': 3})
y = df['Frustrated']

print(X.shape)
X

(112, 9)


,HR_Mean,HR_Median,HR_std,HR_Min,HR_Max,HR_AUC,Phase,Individual,Puzzler
0,4.593227,4.640,3.345290,5.35,3.15,1277.860,3,1,1
1,-2.390863,-2.790,2.517879,-0.76,-2.00,283.315,2,1,1
2,4.544762,3.830,4.054595,1.05,5.70,3318.400,3,1,1
3,2.950165,0.500,6.047603,-0.75,10.93,1950.060,2,1,1
4,3.362441,3.380,3.204755,4.95,2.05,-922.325,3,1,1
...,...,...,...,...,...,...,...,...,...
107,6.450080,10.670,3.921588,3.15,-11.88,1846.265,2,14,0
108,10.549731,5.355,14.549459,1.41,22.19,3348.290,3,14,0
109,15.754642,18.250,9.474556,4.46,19.39,4657.245,2,14,0
110,-6.896583,-4.030,3.589241,1.80,-41.70,-2129.255,3,14,0


In [ ]:
groups = df['Individual']

X_np = X.values.astype(np.float32)
y_np = y.values.astype(np.float32)   

gkf = GroupKFold(n_splits=7)
fold_results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_np, y_np, groups=groups)):
    X_train = torch.tensor(X_np[train_idx])
    X_test  = torch.tensor(X_np[test_idx])
    y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)
    y_test  = torch.tensor(y_np[test_idx]).unsqueeze(1)

    # Here we call a specific instance of the model
    hidden_dim = 50
    input_dim = X_train.shape[1]
    output_dim = 1

    model = get_basic_model(input_dim, hidden_dim, output_dim)

    criterion = torch.nn.MSELoss()

    model, results = fit_nn(model, X_train, y_train, X_test, y_test, criterion, n_epochs=1000, lr=0.01)

    fold_results.append(results)
    print(f"Fold {fold}: final train loss={results['train'][-1]:.4f}, final test loss={results['test'][-1]:.4f}")

Model: Sequential(
  (0): Linear(in_features=9, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=1, bias=True)
)
Epoch [1000/1000], Training loss: nan, Test Loss: nan
Fold 0: final train loss=nan, final test loss=nan
Model: Sequential(
  (0): Linear(in_features=9, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=1, bias=True)
)
Epoch [1000/1000], Training loss: nan, Test Loss: nan
Fold 1: final train loss=nan, final test loss=nan
Model: Sequential(
  (0): Linear(in_features=9, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=1, bias=True)
)
Epoch [1000/1000], Training loss: nan, Test Loss: nan
Fold 2: final train loss=nan, final test loss=nan
Model: Sequential(
  (0): Linear(in_features=9, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=1, bias=True)
)
Epoch [1000/1000], Training loss: nan, Test Loss: nan
Fold 3: final train loss=nan, final te